# 🎯 KMeans — versión mínima

**Idea:** tenemos clientes y queremos **agruparlos en tipos parecidos**. Aquí **no hay respuesta correcta**: los grupos los descubre el modelo.

KMeans coloca **k centros**, une cada cliente al **centro más cercano** y va moviendo los centros hasta que se quedan quietos.

Ejecuta cada celda con **Shift + Enter**, en orden.

## 1. Crear los datos
Esta celda fabrica `clientes.csv` con 300 clientes: `edad`, `ingresos` (miles de €) y `gasto` (de 1 a 100). **No hace falta entenderla.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
perfiles = [(55, 46, 95, 20), (60, 33, 92, 80), (65, 24, 32, 78), (55, 55, 28, 22), (65, 40, 58, 50)]
filas = []
for cantidad, edad, ingresos, gasto in perfiles:
    for _ in range(cantidad):
        filas.append({"edad": np.clip(rng.normal(edad, 6), 18, 70).round(0),
                      "ingresos": np.clip(rng.normal(ingresos, 10), 10, 140).round(0),
                      "gasto": np.clip(rng.normal(gasto, 9), 1, 100).round(0)})
pd.DataFrame(filas).sample(frac=1, random_state=1).to_csv("clientes.csv", index=False)
print("✅ clientes.csv creado")

## 2. Ver los datos
Cada punto es un cliente. ¿Cuántos grupos ves?

In [ ]:
df = pd.read_csv("clientes.csv")

plt.scatter(df["ingresos"], df["gasto"])
plt.xlabel("Ingresos")
plt.ylabel("Gasto")
plt.show()

## 3. Escalar
KMeans mide **distancias**, así que todas las columnas deben estar en la misma escala.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
X = escalador.fit_transform(df)

## 4. Buscar 5 grupos

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
df["grupo"] = kmeans.fit_predict(X)

plt.scatter(df["ingresos"], df["gasto"], c=df["grupo"], cmap="tab10")
plt.xlabel("Ingresos")
plt.ylabel("Gasto")
plt.show()

👀 **Cada color es un grupo.** Solo le hemos dicho cuántos grupos buscar.

## 5. ¿Cuántos grupos? El método del codo
La línea baja rápido y luego **se aplana**. El punto donde se dobla (el "codo") es un buen número de grupos.

In [ ]:
inercias = [KMeans(n_clusters=k, random_state=42).fit(X).inertia_ for k in range(1, 11)]

plt.plot(range(1, 11), inercias, marker="o")
plt.xlabel("Número de grupos (k)")
plt.ylabel("Inercia")
plt.show()

## 6. ¿Cómo es cada grupo?
La media de cada grupo nos ayuda a **ponerles nombre**.

In [ ]:
df.groupby("grupo").mean().round(0)

👀 Por ejemplo: ingresos altos y gasto bajo → *"ahorradores"*; ingresos bajos, gasto alto y jóvenes → *"jóvenes gastadores"*...

## 7. ¿A qué grupo va un cliente nuevo?

In [ ]:
nuevo = pd.DataFrame({"edad": [26], "ingresos": [35], "gasto": [85]})

print("Grupo:", kmeans.predict(escalador.transform(nuevo))[0])

## Resumen
1. **Segmentar = agrupar por parecido**, sin respuesta correcta.
2. **Hay que escalar** y **hay que elegir k** (el codo ayuda).
3. Lo importante es **entender los grupos** y ponerles nombre.

## ✍️ Ejercicios
1. Cambia `n_clusters=5` por 3 en el paso 4. ¿Tienen sentido los grupos?
2. Ponle nombre a cada grupo de la tabla del paso 6.